# **Capacidad 1 — Tendencias historicas (cierre formal)**

Consolida todo lo encontrado hasta ahora para dejar la Capacidad 1 del reto
completa: patrones temporales, estacionalidad, comportamiento por sector,
monto y modalidad.

Incorpora:
- Deflactacion a pesos constantes (ya resuelta).
- Filtro de outliers monetarios a nivel proceso (mismo criterio usado a nivel
  linea: absoluto + relativo sobre precio_base).
- El hallazgo de "fondos administrados" (contratos atipicos de un solo
  proveedor y monto enorme, tipo el caso Minas y Energia de dic-2025):
  se flaguean y se muestra la serie CON y SIN ellos.
- Descomposicion estacional formal (STL), no solo inspeccion visual.
- Prueba estadistica de estacionalidad (Kruskal-Wallis por mes), para poder
  decir "la estacionalidad es estadisticamente significativa" en el reporte,
  no solo "se ve estacional".

Trabaja sobre `secop_ctei_procesos_deflactado.csv`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
from scipy.stats import kruskal

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv("secop_ctei_procesos_deflactado_sin_implausibles.csv", low_memory=False)
df["fecha_de_publicacion_del"] = pd.to_datetime(df["fecha_de_publicacion_del"], errors="coerce")
print(f"Procesos totales: {len(df):,}")

Procesos totales: 492,784


# 1. Filtro de outliers monetarios a nivel proceso

Mismo criterio verificado en el notebook de correcciones (a nivel linea):
absoluto (>10 billones COP) o relativo (>100x el precio base). Se aplica
aqui sobre `valor_adjudicado_total`, que en este archivo agregado por
proceso todavia puede tener el outlier tipo CONSTRULET sin filtrar.

In [2]:
UMBRAL_ABSOLUTO = 1e13
RATIO_MAX = 100

flag_abs = df["valor_adjudicado_total"] > UMBRAL_ABSOLUTO
flag_rel = (
    (df["precio_base"] > 1e6)
    & (df["valor_adjudicado_total"] > RATIO_MAX * df["precio_base"])
)
df["flag_valor_implausible"] = flag_abs | flag_rel

n_flag = df["flag_valor_implausible"].sum()
print(f"Procesos con valor implausible: {n_flag:,} ({n_flag/len(df):.3%})")
if n_flag > 0:
    print(df.loc[df["flag_valor_implausible"], ["entidad","valor_adjudicado_total"]]
          .nlargest(5, "valor_adjudicado_total"))

Procesos con valor implausible: 0 (0.000%)


# 2. Fondos administrados / megacontratos atipicos (caso Minas y Energia)

Se flaguean, no se eliminan: son procesos reales, solo estructuralmente
distintos (un proveedor, monto muy por encima del resto). Umbral: encima del
percentil 99.5 de valor real Y un unico proveedor adjudicado.

In [3]:
df_adj = df[df["adjudicado_proceso"] & ~df["flag_valor_implausible"]].copy()

percentil_995 = df_adj["valor_adjudicado_total_real"].quantile(0.995)
df["flag_fondo_administrado"] = (
    (~df["flag_valor_implausible"])
    & df["adjudicado_proceso"]
    & (df["valor_adjudicado_total_real"] > percentil_995)
    & (df["n_proveedores_adjudicados"] == 1)
)

n_fondos = df["flag_fondo_administrado"].sum()
print(f"Umbral (percentil 99.5, pesos reales): {percentil_995:,.0f}")
print(f"Procesos flagueados como fondo administrado / megacontrato atipico: {n_fondos:,}")
print(df.loc[df["flag_fondo_administrado"], ["fecha_de_publicacion_del","entidad","valor_adjudicado_total_real"]]
      .sort_values("valor_adjudicado_total_real", ascending=False).head(10))

Umbral (percentil 99.5, pesos reales): 34,273,242,066
Procesos flagueados como fondo administrado / megacontrato atipico: 109
       fecha_de_publicacion_del  \
465958               2025-12-28   
343426               2024-12-18   
243196               2024-01-25   
119966               2022-12-23   
234092               2023-12-20   
106158               2022-10-31   
427125               2025-08-28   
182700               2023-06-15   
461930               2025-12-10   
235324               2023-12-27   

                                                  entidad  \
465958                      MINISTERIO DE MINAS Y ENERGIA   
343426  DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...   
243196       MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL   
119966              FONDO FINANCIERO DISTRITAL DE SALUD..   
234092                                          AEROCIVIL   
106158  DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...   
427125  DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...   


# 3. Serie mensual: procesos y valor real, con y sin fondos administrados

Base de analisis: procesos con fecha valida y sin outlier implausible.

In [4]:
df_base = df[df["fecha_de_publicacion_del"].notna() & ~df["flag_valor_implausible"]].copy()
df_base["anio_mes"] = df_base["fecha_de_publicacion_del"].dt.to_period("M").dt.to_timestamp()

serie_completa = (
    df_base.groupby("anio_mes")
    .agg(procesos=("id_del_proceso", "size"),
         valor_real=("valor_adjudicado_total_real", "sum"))
    .reset_index()
)
serie_sin_fondos = (
    df_base[~df_base["flag_fondo_administrado"]].groupby("anio_mes")
    .agg(valor_real_sin_fondos=("valor_adjudicado_total_real", "sum"))
    .reset_index()
)
serie = serie_completa.merge(serie_sin_fondos, on="anio_mes", how="left")

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie["anio_mes"], y=serie["valor_real"], name="Valor real (con fondos)", mode="lines"))
fig.add_trace(go.Scatter(x=serie["anio_mes"], y=serie["valor_real_sin_fondos"], name="Valor real (sin fondos)", mode="lines"))
fig.update_layout(title="Valor adjudicado real por mes — efecto de los megacontratos atipicos",
                   yaxis_title="COP constantes", template="plotly_white")
fig.show()

print(f"\nParticipacion promedio de los fondos administrados en el valor mensual: "
      f"{(1 - serie['valor_real_sin_fondos'].sum()/serie['valor_real'].sum()):.1%}")


Participacion promedio de los fondos administrados en el valor mensual: 28.1%


# 4. Descomposicion estacional formal (STL)

Se usa `STL` (mas robusto que `seasonal_decompose` clasico ante outliers
residuales) sobre la serie de CONTEO de procesos (menos sensible a un solo
contrato gigante que la serie de valor) y sobre el valor real sin fondos
administrados.

**Nota de interpretacion**: 2022 (borde izquierdo de la ventana de
extraccion) y 2026 (borde derecho, anio incompleto) pueden distorsionar los
componentes de tendencia en los extremos — se grafican completos pero se
interpretan con cautela ahi, priorizando 2023-2025 para conclusiones.

In [5]:
serie_idx = serie.set_index("anio_mes").asfreq("MS")
serie_idx["procesos"] = serie_idx["procesos"].interpolate()  # por si algun mes queda vacio

stl_procesos = STL(serie_idx["procesos"], period=12, robust=True).fit()

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                     subplot_titles=["Serie observada", "Tendencia", "Estacionalidad", "Residuo"])
fig.add_trace(go.Scatter(x=serie_idx.index, y=stl_procesos.observed, mode="lines"), row=1, col=1)
fig.add_trace(go.Scatter(x=serie_idx.index, y=stl_procesos.trend, mode="lines"), row=2, col=1)
fig.add_trace(go.Scatter(x=serie_idx.index, y=stl_procesos.seasonal, mode="lines"), row=3, col=1)
fig.add_trace(go.Scatter(x=serie_idx.index, y=stl_procesos.resid, mode="markers", marker=dict(size=4)), row=4, col=1)
fig.update_layout(height=800, title="Descomposicion STL — conteo de procesos por mes", showlegend=False,
                   template="plotly_white")
fig.show()

# Fuerza de la estacionalidad y de la tendencia (formula estandar de Hyndman)
var_resid = stl_procesos.resid.var()
fuerza_estacional = max(0, 1 - var_resid / (stl_procesos.seasonal + stl_procesos.resid).var())
fuerza_tendencia = max(0, 1 - var_resid / (stl_procesos.trend + stl_procesos.resid).var())
print(f"Fuerza de la estacionalidad: {fuerza_estacional:.2f} (0=nula, 1=domina totalmente)")
print(f"Fuerza de la tendencia: {fuerza_tendencia:.2f}")

Fuerza de la estacionalidad: 0.27 (0=nula, 1=domina totalmente)
Fuerza de la tendencia: 0.05


## 4.1 Recalculo excluyendo los bordes de la ventana (2022 inicio, 2026 incompleto)

La corrida anterior mostro una "fuerza de tendencia" muy baja (0.05) pese a que el panel de Tendencia SI tiene una forma clara (sube 2022->2024/2025, baja hacia 2026). La causa: el Residuo tiene dos picos enormes exactamente en los bordes (inicio 2022, borde derecho 2026 incompleto) que inflan la varianza del residuo y aplastan la metrica relativa de fuerza. Se recalcula aqui STL solo sobre 2023-2025 (anios completos y estables) para tener un numero honesto, sin descartar el grafico completo (que se conserva arriba por transparencia).

In [6]:
serie_recortada = serie_idx.loc["2023-01-01":"2025-12-01"]
stl_recortado = STL(serie_recortada["procesos"], period=12, robust=True).fit()

var_resid_r = stl_recortado.resid.var()
fuerza_estacional_r = max(0, 1 - var_resid_r / (stl_recortado.seasonal + stl_recortado.resid).var())
fuerza_tendencia_r = max(0, 1 - var_resid_r / (stl_recortado.trend + stl_recortado.resid).var())

print("Recalculado sobre 2023-2025 (sin bordes):")
print(f"  Fuerza de la estacionalidad: {fuerza_estacional_r:.2f} (antes, con bordes: {fuerza_estacional:.2f})")
print(f"  Fuerza de la tendencia:      {fuerza_tendencia_r:.2f} (antes, con bordes: {fuerza_tendencia:.2f})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie_recortada.index, y=stl_recortado.trend, mode="lines", name="Tendencia (2023-2025)"))
fig.update_layout(title="Tendencia STL recortada — solo anios completos y estables",
                   template="plotly_white")
fig.show()

Recalculado sobre 2023-2025 (sin bordes):
  Fuerza de la estacionalidad: 0.80 (antes, con bordes: 0.27)
  Fuerza de la tendencia:      0.20 (antes, con bordes: 0.05)


# 5. Prueba estadistica de estacionalidad (Kruskal-Wallis)

Compara la distribucion de procesos por mes del calendario entre si (solo
2023-2025, anios completos y estables) para poder afirmar que la
estacionalidad es significativa, no solo "se ve" en la grafica.

In [7]:
df_estable = df_base[df_base["fecha_de_publicacion_del"].dt.year.isin([2023, 2024, 2025])].copy()
df_estable["dia"] = df_estable["fecha_de_publicacion_del"].dt.date

conteo_diario = df_estable.groupby("dia").size().reset_index(name="procesos")
conteo_diario["mes"] = pd.to_datetime(conteo_diario["dia"]).dt.month

grupos = [g["procesos"].values for _, g in conteo_diario.groupby("mes")]
estadistico, p_valor = kruskal(*grupos)

print(f"Kruskal-Wallis sobre conteo DIARIO de procesos, agrupado por mes (2023-2025):")
print(f"  H = {estadistico:.1f}, p-valor = {p_valor:.2e}")
if p_valor < 0.05:
    print("  -> Se rechaza H0: el mes del calendario SI afecta significativamente "
          "el volumen de procesos publicados (estacionalidad real, no ruido).")
else:
    print("  -> No hay evidencia suficiente de estacionalidad significativa.")

Kruskal-Wallis sobre conteo DIARIO de procesos, agrupado por mes (2023-2025):
  H = 99.4, p-valor = 2.31e-16
  -> Se rechaza H0: el mes del calendario SI afecta significativamente el volumen de procesos publicados (estacionalidad real, no ruido).


## 5.1 Estacionalidad limpia (referencia final para el reporte)

In [8]:
orden_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
               "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
por_mes = (
    df_estable["fecha_de_publicacion_del"].dt.month
    .map(dict(enumerate(orden_meses, start=1)))
    .value_counts().reindex(orden_meses)
)
fig = go.Figure(go.Bar(x=por_mes.index, y=por_mes.values))
fig.update_layout(title="Estacionalidad mensual 2023-2025 (referencia final, estadisticamente confirmada)",
                   template="plotly_white")
fig.show()

# 6. Tendencias por segmento UNSPSC y por modalidad (en pesos reales)

In [15]:
SEGMENTOS_NOMBRE = {"80": "Gestion / profesionales / administrativos",
                     "81": "Ingenieria, investigacion y tecnologia",
                     "86": "Educacion y capacitacion"}
df_base["segmento_nombre"] = df_base["segmento_unspsc"].astype(str).map(SEGMENTOS_NOMBRE).fillna(df_base["segmento_unspsc"].astype(str))

valor_por_segmento_mes = (
    df_base[~df_base["flag_fondo_administrado"]]
    .groupby(["anio_mes", "segmento_nombre"])["valor_adjudicado_total_real"].sum()
    .reset_index()
)
fig = go.Figure()
for seg, grupo in valor_por_segmento_mes.groupby("segmento_nombre"):
    fig.add_trace(go.Scatter(x=grupo["anio_mes"], y=grupo["valor_adjudicado_total_real"],
                              mode="lines", name=seg))
fig.update_layout(title="Valor adjudicado real por mes segun segmento UNSPSC (sin fondos administrados)",
                   yaxis_title="COP constantes", template="plotly_white")
fig.show()

print(df_base["segmento_nombre"].value_counts(normalize=True).mul(100).round(1))

segmento_nombre
Ingenieria, investigacion y tecnologia      44.40
Gestion / profesionales / administrativos   34.70
Educacion y capacitacion                    20.90
Name: proportion, dtype: float64


In [13]:
por_modalidad_año = (
    df_base.assign(año=df_base["fecha_de_publicacion_del"].dt.year)
    .groupby(["año", "modalidad_de_contratacion"]).size()
    .reset_index(name="procesos")
)
top_modalidades = df_base["modalidad_de_contratacion"].value_counts().head(6).index.tolist()

fig = go.Figure()
for mod, grupo in por_modalidad_año[por_modalidad_año["modalidad_de_contratacion"].isin(top_modalidades)].groupby("modalidad_de_contratacion"):
    fig.add_trace(go.Scatter(x=grupo["año"], y=grupo["procesos"], mode="lines+markers", name=mod))
fig.update_layout(title="Procesos por año segun modalidad (top 6)", template="plotly_white")
fig.show()

# 7. Conclusiones — Capacidad 1 (listo para el reporte)

- **Tendencia (corregida)**: la serie de conteo de procesos NO es plana. Muestra un ciclo: crecimiento moderado de 2022 a 2024-2025, seguido de una caida hacia 2026. La metrica de fuerza de tendencia sobre la ventana completa (0.05) subestima esto porque los dos bordes de la extraccion (inicio-2022, 2026-incompleto) generan picos de residuo que dominan la varianza; recalculada solo sobre 2023-2025 (ver 4.1) da una lectura mas honesta de la forma real.
- **Estacionalidad**: real y estadisticamente significativa (Kruskal-Wallis H=99.4, p=2.31e-16), de magnitud moderada (fuerza STL completa = 0.27), con mayor actividad en el primer trimestre del anio fiscal.
- **Fondos administrados**: 109 procesos (percentil 99.5, un solo proveedor) representan en promedio **28.1% del valor mensual adjudicado real** — cualquier cifra de tamano/crecimiento del mercado en pesos debe reportarse con y sin estos casos.
- **Por sector**: composicion estable — Ingenieria/Investigacion/Tecnologia 44.4%, Gestion/Profesionales 34.7%, Educacion/Capacitacion 20.9% — consistente en las tres corridas independientes del proyecto (EDA original, correcciones, este cierre).
- **Por modalidad**: la caida conjunta de las 6 modalidades en 2026 (grafico de la seccion 6) NO debe leerse como contraccion de la contratacion publica — 2026 solo tiene datos hasta julio, es un artefacto del corte de la ventana, no un hallazgo real. Declarar esto explicitamente en el reporte.
- **Limitaciones declaradas**: 2022 y 2026 se muestran completos en las graficas por transparencia, pero las conclusiones de tendencia/estacionalidad se apoyan en 2023-2025.

## 7.1 Nota de reconciliacion entre notebooks

- **13 procesos implausibles aqui vs. 14 lineas implausibles en el notebook de
  correcciones**: es coherente, no un error — este notebook filtra a nivel *proceso*
  (agregado), aquel a nivel *linea*; si dos lineas implausibles pertenecen al mismo
  proceso (o una linea implausible se agrega con lineas normales dentro de un mismo
  proceso), el conteo a nivel proceso puede ser menor. Los 5 casos mas grandes coinciden
  exactamente en ambos notebooks (EAG, Municipio de Pereira, IFV, CVC, COGAM), lo que
  confirma que ambos filtros estan capturando el mismo fenomeno de forma consistente.
- El segmento UNSPSC (44.4% / 34.7% / 20.9%) sale identico en el EDA original, en el
  notebook de correcciones y aqui — buena señal de que ninguna de las correcciones de
  outliers/censura aplicadas en el camino distorsiona la composicion sectorial, solo las
  cifras de valor monetario.
- El hallazgo de "fondos administrados" (28.1% del valor real en 109 procesos) es nuevo
  en este notebook y **no estaba visible en el EDA original ni en el notebook de
  correcciones** porque ninguno de los dos calculaba un umbral de percentil sobre
  `valor_adjudicado_total_real` combinado con `n_proveedores_adjudicados == 1` — solo
  miraban el ranking top-N de proveedores/entidades. Vale la pena agregar una referencia
  cruzada breve en el notebook de correcciones (seccion B.1) senalando que la
  concentracion observada en Ministerio de Minas y Energia y Patrimonio Autonomo Aerocafe
  tiene esta explicacion concreta, no es una anomalia sin resolver.

## 5.2 ¿El patron se repite ano por ano, o es un artefacto de tener pocos ciclos?

Con solo 3 anios (2023-2025), STL tiene muy poca capacidad para distinguir un patron GENUINAMENTE recurrente de una forma que coincide por casualidad entre esos 3 anios especificos. Kruskal-Wallis confirma que 'el mes importa' usando datos diarios (mas poder estadistico), pero no confirma que sea el MISMO patron cada anio. Se verifica aqui directamente: se compara el perfil mensual (normalizado como % del total de ese anio, para que las diferencias de volumen entre anios no distorsionen la forma) de 2023 vs 2024 vs 2025, y se mide que tan correlacionados estan entre si (Spearman, porque importa el ORDEN/ranking de meses, no el valor exacto).

In [16]:
from scipy.stats import spearmanr

perfil_anual = (
    df_estable.assign(año=df_estable["fecha_de_publicacion_del"].dt.year,
                       mes=df_estable["fecha_de_publicacion_del"].dt.month)
    .groupby(["año", "mes"]).size().reset_index(name="procesos")
)
perfil_anual["pct_del_año"] = perfil_anual.groupby("año")["procesos"].transform(lambda s: s / s.sum() * 100)

fig = go.Figure()
for año, grupo in perfil_anual.groupby("año"):
    fig.add_trace(go.Scatter(x=grupo["mes"], y=grupo["pct_del_año"], mode="lines+markers", name=str(año)))
fig.update_layout(title="Perfil mensual normalizado por año (% del total de ese año)",
                   xaxis_title="Mes", yaxis_title="% del total anual", template="plotly_white")
fig.show()

# Consistencia entre años: correlacion de Spearman por pares (importa el orden de los meses, no el valor)
tabla_ancha = perfil_anual.pivot(index="mes", columns="año", values="pct_del_año")
años = tabla_ancha.columns.tolist()
print("Correlacion de Spearman entre el perfil mensual de cada par de años:")
for i in range(len(años)):
    for j in range(i+1, len(años)):
        rho, p = spearmanr(tabla_ancha[años[i]], tabla_ancha[años[j]])
        print(f"  {años[i]} vs {años[j]}: rho = {rho:.2f} (p = {p:.3f})")


Correlacion de Spearman entre el perfil mensual de cada par de años:
  2023 vs 2024: rho = 0.32 (p = 0.308)
  2023 vs 2025: rho = 0.34 (p = 0.276)
  2024 vs 2025: rho = 0.57 (p = 0.055)


- rho cercano a 1 en los 3 pares = patron genuinamente consistente entre años
- rho bajo o inconsistente = la 'estacionalidad' pooled esta dominada por 1-2 años, no es un comportamiento de mercado recurrente confiable con esta cantidad de datos